In [9]:
%pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [10]:
# sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Gerar dados
x, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=2,
    n_redundant=2,
    random_state=54
)

# 2. Split
train_samples = 100 
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    train_size=train_samples,
    random_state=54
)

# 3. Modelo
model = LogisticRegression(max_iter=1000)

# 4. Treino
model.fit(x_train, y_train)

# 5. Predição
y_pred = model.predict(x_test)

# 6. Avaliação
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

# 7. Output bonito
print("=" * 50)
print("📊 RESULTADOS DO MODELO")
print("=" * 50)

print(f"\n🔹 Tamanho do treino: {len(x_train)}")
print(f"🔹 Tamanho do teste: {len(x_test)}")

print("\n🎯 Accuracy:")
print(f"   {accuracy * 100:.2f}%")

print("\n📉 Matriz de Confusão:")
print(conf_matrix)

print("\n📄 Classification Report:")
print(report)

print("=" * 50)

📊 RESULTADOS DO MODELO

🔹 Tamanho do treino: 100
🔹 Tamanho do teste: 900

🎯 Accuracy:
   98.33%

📉 Matriz de Confusão:
[[437  14]
 [  1 448]]

📄 Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98       451
           1       0.97      1.00      0.98       449

    accuracy                           0.98       900
   macro avg       0.98      0.98      0.98       900
weighted avg       0.98      0.98      0.98       900



In [11]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [13]:
# TensorFlow
import tensorflow as tf

# 1. Gerar dados sintéticos manualmente
num_samples = 1000
num_features = 20

x = tf.random.normal((num_samples, num_features))
true_weights = tf.random.normal((num_features, 1))

# função linear + ruído
logits = tf.matmul(x, true_weights) + tf.random.normal((num_samples, 1)) * 0.5

# transformar em classificação binária
y = tf.cast(logits > 0, tf.float32)

# 2. Split manual (80/20)
train_size = int(0.8 * num_samples)

x_train = x[:train_size]
x_test = x[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]

# 3. Normalização manual (mean/std do treino)
mean = tf.reduce_mean(x_train, axis=0)
std = tf.math.reduce_std(x_train, axis=0)

x_train = (x_train - mean) / (std + 1e-7)
x_test = (x_test - mean) / (std + 1e-7)

# 4. Dataset API (mais próximo de produção)
batch_size = 32

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(batch_size)

test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_dataset = test_dataset.batch(batch_size)

# 5. Modelo
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(num_features,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# 6. Compilar
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 7. Treinar
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data=test_dataset,
    verbose=0
)

# 8. Avaliação manual
y_pred_prob = model.predict(x_test)
y_pred = tf.cast(y_pred_prob > 0.5, tf.float32)

# accuracy manual
accuracy = tf.reduce_mean(tf.cast(tf.equal(y_pred, y_test), tf.float32))

# 9. Output organizado
print("=" * 50)
print("🧠 RESULTADOS (TensorFlow PURO)")
print("=" * 50)

print(f"\n🔹 Treino: {x_train.shape[0]} amostras")
print(f"🔹 Teste: {x_test.shape[0]} amostras")

print("\n🎯 Accuracy (manual):")
print(f"   {accuracy * 100:.2f}%")

print("\n📊 Últimas épocas:")
for i in range(-5, 0):
    print(f"Epoch {len(history.history['loss']) + i + 1}: "
          f"loss={history.history['loss'][i]:.4f} | "
          f"val_loss={history.history['val_loss'][i]:.4f}")

print("=" * 50)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
🧠 RESULTADOS (TensorFlow PURO)

🔹 Treino: 800 amostras
🔹 Teste: 200 amostras

🎯 Accuracy (manual):
   93.00%

📊 Últimas épocas:
Epoch 16: loss=0.0828 | val_loss=0.1810
Epoch 17: loss=0.0751 | val_loss=0.1802
Epoch 18: loss=0.0678 | val_loss=0.1785
Epoch 19: loss=0.0627 | val_loss=0.1772
Epoch 20: loss=0.0573 | val_loss=0.1776
